# Probabilistic roadmaps (PRM): notebook

The room of RO-009, the robot's 0.25 m disk footprint and its obstacle region of RO-037 (`images/cspace.py`).
The roadmaps of this Note are in `images/prm.py`. Here we compute every number of the Note. Seeds are fixed.

In [1]:
import sys
sys.path.insert(0, 'images')
from multiprocessing import Pool
import numpy as np
import shapely
from shapely.geometry import Point, box
import cspace as c
import prm
w = prm.World()          # the 0.25 m disk

## 1. The room, the four trips and the cost of a grid (Note, Section 1)

In [2]:
for a, s, b, g in prm.TRIPS:
    print(f'{a:8s} -> {b:8s}', s, '->', g, ' free ends:', w.free(s, False), w.free(g, False),
          ' straight line', round(float(np.hypot(g[0] - s[0], g[1] - s[1])), 3), 'm')
free = box(0, 0, c.W, c.H).difference(w.cobs)
print('free floor for the centre:', round(free.area, 2), 'm2 of', c.W * c.H, 'm2:', round(free.area / (c.W * c.H), 3))
for d in (2, 3, 7):
    print(f'grid with 10 values per joint, {d} joints: {10 ** d:,} cells')

dock     -> door     (2.0, 1.0) -> (3.6, 4.0)  free ends: True True  straight line 3.4 m
door     -> cabinet  (3.6, 4.0) -> (5.9, 2.0)  free ends: True True  straight line 3.048 m
cabinet  -> corner   (5.9, 2.0) -> (6.6, 0.4)  free ends: True True  straight line 1.746 m
corner   -> dock     (6.6, 0.4) -> (2.0, 1.0)  free ends: True True  straight line 4.639 m
free floor for the centre: 15.02 m2 of 28.0 m2: 0.536
grid with 10 values per joint, 2 joints: 100 cells
grid with 10 values per joint, 3 joints: 1,000 cells
grid with 10 values per joint, 7 joints: 10,000,000 cells


## 2. Building the roadmap and the first trip (Note, Section 2)

In [3]:
rng = np.random.default_rng(0)
w.checks = 0
log = []
V = prm.sample_free(w, rng, 100, log)
phase1 = w.checks
elog = []
E = prm.connect(w, V, prm.K, elog)
phase2 = w.checks - phase1
nc, lab = prm.components(len(V), E)
print('draws', len(log), ' in collision', sum(not ok for _, ok in log), ' milestones', len(V))
print('edges tried', len(elog), ' added', len(E), ' rejected', len(elog) - len(E))
print('checks: phase 1', phase1, ' phase 2', phase2, ' build total', phase1 + phase2)
print('pieces of the roadmap', nc)
print('first milestone', np.round(V[0], 3), ' its nearest 10 tried edges:',
      [(j, ok) for i, j, ok, _ in elog if i == 0][:10])

draws 189  in collision 89  milestones 100
edges tried 573  added 525  rejected 48
checks: phase 1 189  phase 2 7745  build total 7934
pieces of the roadmap 1
first milestone [3.805 3.74 ]  its nearest 10 tried edges: [(np.int64(37), True), (np.int64(58), True), (np.int64(4), True), (np.int64(80), True), (np.int64(81), True), (np.int64(16), True), (np.int64(21), True), (np.int64(60), True), (np.int64(11), True), (np.int64(30), True)]


In [4]:
qlog = []
w0 = w.checks
P, L, extra = prm.query(w, V, E, prm.START, prm.DOOR, log=qlog)
print('trip 1 dock -> door: length', round(L, 3), 'm, vertices', len(P), ', edges tried', len(qlog),
      ', added', len(extra), ', checks', w.checks - w0)
print('straight drive (RO-037): 3.4 m')

trip 1 dock -> door: length 3.531 m, vertices 6 , edges tried 20 , added 18 , checks 226
straight drive (RO-037): 3.4 m


A denser roadmap gives a shorter route (same sample stream, seed 0).

In [5]:
rng = np.random.default_rng(0)
Vbig = prm.sample_free(prm.World(), rng, 400)
for n in (20, 50, 100, 200, 400):
    ww = prm.World()
    Eb = prm.connect(ww, Vbig[:n], prm.K)
    Pb, Lb, _ = prm.query(ww, Vbig[:n], Eb, prm.START, prm.DOOR)
    print(n, 'milestones: dock -> door', round(Lb, 3), 'm')

20 milestones: dock -> door 3.4 m
50 milestones: dock -> door 3.694 m
100 milestones: dock -> door 3.531 m


200 milestones: dock -> door 3.445 m


400 milestones: dock -> door 3.53 m


## 3. Many trips on one roadmap (Note, Section 3)

In [6]:
build = phase1 + phase2
trip_cost = []
for a, s, b, g in prm.TRIPS:
    w0 = w.checks
    P, L, extra = prm.query(w, V, E, s, g)
    trip_cost.append(w.checks - w0)
    print(f'{a:8s} -> {b:8s}: {L:.3f} m, {len(P) - 2} milestones on the route, {w.checks - w0} checks')
print('one roadmap, four trips:', build, '+', sum(trip_cost), '=', build + sum(trip_cost))
print('a new roadmap for every trip:', 4 * build + sum(trip_cost))
print('share of one trip in the build: up to', round(max(trip_cost) / build, 3))

dock     -> door    : 3.531 m, 4 milestones on the route, 226 checks
door     -> cabinet : 3.587 m, 4 milestones on the route, 209 checks
cabinet  -> corner  : 1.898 m, 2 milestones on the route, 195 checks
corner   -> dock    : 4.758 m, 5 milestones on the route, 212 checks
one roadmap, four trips: 7934 + 842 = 8776
a new roadmap for every trip: 32578
share of one trip in the build: up to 0.028


## 4. Accessibility and connectivity (Note, Section 4)

In [7]:
for n in (5, 10, 20, 50, 100):
    En = prm.connect(prm.World(), V[:n], prm.K)
    ncn, labn = prm.components(n, En)
    acc, Pg, a = prm.accessible_share(w, V[:n], prm.K, 0.1)
    ok = [prm.query(prm.World(), V[:n], En, s, g)[0] is not None for _, s, _, g in prm.TRIPS]
    print(f'{n:3d} milestones: {len(En):3d} edges, {ncn} piece(s) {np.bincount(labn).tolist()}, '
          f'free floor that reaches a milestone {acc:.3f}, trips found {ok}')

  5 milestones:   3 edges, 2 piece(s) [3, 2], free floor that reaches a milestone 0.982, trips found [True, False, True, False]
 10 milestones:  16 edges, 2 piece(s) [6, 4], free floor that reaches a milestone 0.997, trips found [True, False, True, False]
 20 milestones:  78 edges, 1 piece(s) [20], free floor that reaches a milestone 0.999, trips found [True, True, True, True]


 50 milestones: 246 edges, 1 piece(s) [50], free floor that reaches a milestone 1.000, trips found [True, True, True, True]


100 milestones: 525 edges, 1 piece(s) [100], free floor that reaches a milestone 1.000, trips found [True, True, True, True]


## 5. Completeness (Note, Section 5)
Grids of configurations: the robot's centre at the middle of every cell, kept if the disk is free there; neighbours
(8 directions) joined by free straight segments; each end of a trip joined to the grid points it sees within 1.5 cells.

In [8]:
from scipy.spatial import cKDTree
def grid_plan(h, s, g):
    xs, ys = np.arange(h / 2, c.W, h), np.arange(h / 2, c.H, h)
    Pg = np.array([(x, y) for y in ys for x in xs])
    total = len(Pg)
    Pg = Pg[w.free_xy(Pg[:, 0], Pg[:, 1])]
    idx = {tuple(np.round(p, 6)): i for i, p in enumerate(Pg)}
    Eg = []
    for i, p in enumerate(Pg):
        for dx, dy in ((h, 0), (0, h), (h, h), (h, -h)):
            q = tuple(np.round(p + np.array([dx, dy]), 6))
            if q in idx and w.seg_free_exact(p, q):
                Eg.append((i, idx[q]))
    Vg = [tuple(p) for p in Pg] + [s, g]
    tree = cKDTree(Pg)
    for qi, q in ((len(Pg), s), (len(Pg) + 1, g)):
        for j in tree.query_ball_point(q, h * 1.5):
            if w.seg_free_exact(q, Pg[j]):
                Eg.append((qi, j))
    path, L, _ = prm.astar(Vg, Eg, len(Pg), len(Pg) + 1)
    return total, len(Pg), L
for h in (0.5, 0.25):
    out = [grid_plan(h, s, g) for _, s, _, g in prm.TRIPS]
    print(f'grid {h} m: {out[0][1]} of {out[0][0]} points free; trips:', [round(o[2], 3) for o in out])

grid 0.5 m: 44 of 112 points free; trips: [3.698, 3.497, inf, inf]
grid 0.25 m: 244 of 448 points free; trips: [3.676, 3.447, 1.903, 4.86]


In [9]:
def success(args):
    n, seed = args
    ww = prm.World()
    rng = np.random.default_rng(1000 + seed)
    Vs, Es = prm.build(ww, rng, n)
    found = [prm.query(ww, Vs, Es, s, g)[0] is not None for _, s, _, g in prm.TRIPS]
    segs = shapely.linestrings(np.stack([np.asarray(Vs), np.tile(prm.CORNER, (n, 1))], 1))
    sees = bool((~shapely.intersects(ww.cobs, segs)).any())
    return found + [sees]
p_corner = prm.visible_share(w, prm.CORNER)[0]
print('share of the free floor that sees the corner: p =', round(p_corner, 4))
for name, q in (('door', prm.DOOR), ('cabinet', prm.CABINET)):
    print('share of the free floor that sees the', name, round(prm.visible_share(w, q)[0], 4))
print('0.99 ** 100 =', round(0.99 ** 100, 4), ' 0.99 ** 300 =', round(0.99 ** 300, 4))
rates = {}
with Pool(10) as pool:
    for n in (5, 10, 20, 50, 100):
        r = np.array(pool.map(success, [(n, s) for s in range(200)]), float).mean(0)
        rates[n] = r
        print(f'{n:3d} milestones, 200 runs: trips found {np.round(r[:4], 3)}; corner sees a milestone {r[4]:.3f}, '
              f'formula 1 - (1 - p)^n = {1 - (1 - p_corner) ** n:.3f}')

share of the free floor that sees the corner: p = 0.2795
share of the free floor that sees the door 0.5788
share of the free floor that sees the cabinet 0.1701
0.99 ** 100 = 0.366  0.99 ** 300 = 0.049


  5 milestones, 200 runs: trips found [0.965 0.14  0.525 0.53 ]; corner sees a milestone 0.820, formula 1 - (1 - p)^n = 0.806


 10 milestones, 200 runs: trips found [1.    0.375 0.765 0.83 ]; corner sees a milestone 0.970, formula 1 - (1 - p)^n = 0.962


 20 milestones, 200 runs: trips found [1.    0.715 0.945 0.915]; corner sees a milestone 1.000, formula 1 - (1 - p)^n = 0.999


 50 milestones, 200 runs: trips found [1.   0.97 1.   0.99]; corner sees a milestone 1.000, formula 1 - (1 - p)^n = 1.000


100 milestones, 200 runs: trips found [1.    0.995 1.    1.   ]; corner sees a milestone 1.000, formula 1 - (1 - p)^n = 1.000


In [10]:
for n in (1, 2, 3, 5, 10, 20):
    print(n, 'milestones: all miss the corner with chance', round((1 - p_corner) ** n, 4))
print('0.72 ** 5 =', round(0.72 ** 5, 4), ' 0.72 ** 10 =', round(0.72 ** 10, 4))

1 milestones: all miss the corner with chance 0.7205
2 milestones: all miss the corner with chance 0.5192
3 milestones: all miss the corner with chance 0.3741
5 milestones: all miss the corner with chance 0.1942
10 milestones: all miss the corner with chance 0.0377
20 milestones: all miss the corner with chance 0.0014
0.72 ** 5 = 0.1935  0.72 ** 10 = 0.0374


## 6. Narrow passages (Note, Section 6): the robot with a round tray 0.8 m across (radius 0.4 m)

In [11]:
wt = prm.World(0.4)
T = Point(3.0, 1.0)
print('corridor under the table: 1.0 - 2 x 0.4 =', round(1.0 - 0.8, 3), 'm')
print('table corner (5.5, 2.5) to cabinet (6.4, 2.5):', 0.9, '-> gap', round(0.9 - 0.8, 3),
      '; to cabinet corner (5.6, 3.4):', round(float(np.hypot(0.1, 0.9)), 3), '-> gap', round(float(np.hypot(0.1, 0.9)) - 0.8, 3))
d_pole = shapely.geometry.LineString([(5, 1), (5.5, 2.5)]).distance(Point(6, 0.8))
print('table edge to green pole centre', round(d_pole, 3), '-> gap', round(d_pole - 0.4 - 0.5, 3))
freet = box(0, 0, c.W, c.H).difference(wt.cobs)
print('free floor for the centre with the tray:', round(freet.area, 2), 'm2; pieces',
      len(getattr(freet, 'geoms', [freet])), '; dock and cabinet spot free:', wt.free(prm.START, False), wt.free(prm.CABINET, False))
corridor = box(3.0, 0.4, 5.0, 0.6)
print('corridor area', round(corridor.area, 3), 'm2 =', round(corridor.area / freet.area, 4), 'of the free floor')

corridor under the table: 1.0 - 2 x 0.4 = 0.2 m
table corner (5.5, 2.5) to cabinet (6.4, 2.5): 0.9 -> gap 0.1 ; to cabinet corner (5.6, 3.4): 0.906 -> gap 0.106
table edge to green pole centre 1.012 -> gap 0.112
free floor for the centre with the tray: 9.96 m2; pieces 1 ; dock and cabinet spot free: True True
corridor area 0.4 m2 = 0.0402 of the free floor


In [12]:
def narrow(args):
    strategy, n, sigma, seed, r = args
    ww = prm.World(r)
    rng = np.random.default_rng(5000 + seed)
    Vs, draws = prm.sample_strategy(ww, rng, n, strategy, sigma, 1.0)
    Es = prm.connect(ww, Vs, prm.K)
    P, L, _ = prm.query(ww, Vs, Es, prm.START, prm.CABINET)
    return P is not None, draws, ww.checks
found = {}
with Pool(10) as pool:
    for strategy, n, sigma, r in [('uniform', 50, 0, 0.25), ('uniform', 50, 0, 0.4), ('uniform', 100, 0, 0.4),
                                  ('uniform', 200, 0, 0.4), ('gaussian', 50, 0.1, 0.4), ('bridge', 50, 0.3, 0.4)]:
        res = np.array(pool.map(narrow, [(strategy, n, sigma, s, r) for s in range(100)]), float)
        found[strategy, n, r] = res[:, 0].astype(bool)
        print(f'r = {r}: {strategy:8s} {n:3d} milestones (sigma {sigma}): dock -> cabinet found in {res[:, 0].mean():.2f} '
              f'of 100 runs; draws {res[:, 1].mean():.0f}; checks {res[:, 2].mean():.0f}')
u, g, b = found['uniform', 50, 0.4], found['gaussian', 50, 0.4], found['bridge', 50, 0.4]
from collections import Counter
outcomes = Counter(zip(u, g, b))
print('outcomes (uniform, gaussian, bridge found) over the 100 runs:', dict(outcomes))
print('uniform fails, both others succeed:', int(np.sum(~u & g & b)), 'runs; seed 5001 (run 1):', u[1], g[1], b[1])


r = 0.25: uniform   50 milestones (sigma 0): dock -> cabinet found in 0.93 of 100 runs; draws 94; checks 5490


r = 0.4: uniform   50 milestones (sigma 0): dock -> cabinet found in 0.32 of 100 runs; draws 142; checks 4529


r = 0.4: uniform  100 milestones (sigma 0): dock -> cabinet found in 0.66 of 100 runs; draws 284; checks 6983


r = 0.4: uniform  200 milestones (sigma 0): dock -> cabinet found in 0.93 of 100 runs; draws 566; checks 10660


r = 0.4: gaussian  50 milestones (sigma 0.1): dock -> cabinet found in 0.86 of 100 runs; draws 536; checks 5311


r = 0.4: bridge    50 milestones (sigma 0.3): dock -> cabinet found in 0.93 of 100 runs; draws 2470; checks 8881
outcomes (uniform, gaussian, bridge found) over the 100 runs: {(np.False_, np.True_, np.False_): 4, (np.False_, np.True_, np.True_): 53, (np.True_, np.True_, np.True_): 27, (np.True_, np.True_, np.False_): 2, (np.False_, np.False_, np.False_): 1, (np.False_, np.False_, np.True_): 10, (np.True_, np.False_, np.True_): 3}
uniform fails, both others succeed: 53 runs; seed 5001 (run 1): False True True


One run of each strategy (seed 5001, run 1 of the 100 above), drawn in Figure 8.

In [13]:
for strategy, sigma in (('uniform', 0), ('gaussian', 0.1), ('bridge', 0.3)):
    ww = prm.World(0.4)
    rng = np.random.default_rng(5001)
    Vs, draws = prm.sample_strategy(ww, rng, 50, strategy, sigma, 1.0)
    Es = prm.connect(ww, Vs, prm.K)
    ncs, labs = prm.components(50, Es)
    P, L, _ = prm.query(ww, Vs, Es, prm.START, prm.CABINET)
    near = sum(1 for v in Vs if wt.cobs.distance(Point(v)) < 0.15)
    sizes = sorted(np.bincount(labs).tolist(), reverse=True)
    print(f'{strategy:8s}: {draws} draws, {ncs} pieces {sizes}, route', None if P is None else round(L, 3),
          f'; milestones within 0.15 m of an obstacle: {near}')

uniform : 165 draws, 2 pieces [45, 5], route None ; milestones within 0.15 m of an obstacle: 23
gaussian: 475 draws, 1 pieces [50], route 4.993 ; milestones within 0.15 m of an obstacle: 49


bridge  : 2152 draws, 2 pieces [47, 3], route 7.773 ; milestones within 0.15 m of an obstacle: 48


In [14]:
# Figure 9: Gaussian sampling (seed 22) and the bridge test (seed 2), draw by draw
ww = prm.World(0.4)
rng = np.random.default_rng(22)
G = []
while sum(k is not None for k, _, _ in G) < 20:
    G.append(prm.gaussian_sample(ww, rng, 0.1))
k, q1, q2 = G[2]
print('Gaussian: 20 milestones from', len(G), 'draws; draw 3: q1', np.round(q1, 3), ww.free(q1, False),
      'q2', np.round(q2, 3), ww.free(q2, False), 'kept', k is not None)
rng = np.random.default_rng(2)
B = []
while sum(m is not None for m, _, _, _ in B) < 10:
    B.append(prm.bridge_sample(ww, rng, 0.3))
m, q1, q2, mid = B[3]
print('bridge test: 10 milestones from', len(B), 'draws; draw 4: q1', np.round(q1, 4), 'q2', np.round(q2, 4),
      'midpoint', np.round(mid, 4), 'kept', m is not None)

Gaussian: 20 milestones from 144 draws; draw 3: q1 [0.36  2.221] False q2 [0.407 2.343] True kept True


bridge test: 10 milestones from 460 draws; draw 4: q1 [3.9359 0.6002] q2 [3.8372 0.3626] midpoint [3.8865 0.4814] kept True


## 7. The visibility roadmap (Note, Section 7)

In [15]:
wv = prm.World()
rng = np.random.default_rng(0)
vlog = []
G, C, Ev, comp = prm.visibility_roadmap(wv, rng, 300, vlog)
kinds = [k for k, _, _ in vlog]
print('300 draws:', {k: kinds.count(k) for k in ('collision', 'guard', 'connector', 'discarded')})
print('guards', len(G), ' connectors', len(C), ' edges', len(Ev), ' pieces', len(set(comp)), ' checks', wv.checks)
print('last new guard or connector at draw', max(i for i, k in enumerate(kinds) if k in ('guard', 'connector')) + 1)
Vv = G + C
idx = {tuple(v): i for i, v in enumerate(Vv)}
EEv = [(idx[tuple(a)], idx[tuple(b)]) for a, b in Ev]
for a, s, b, g in prm.TRIPS:
    P, L, _ = prm.query(prm.World(), Vv, EEv, s, g, k=len(Vv))
    print(f'{a:8s} -> {b:8s}:', None if P is None else round(L, 3), 'm')

300 draws: {'collision': 141, 'guard': 5, 'connector': 4, 'discarded': 150}
guards 5  connectors 4  edges 8  pieces 1  checks 10513
last new guard or connector at draw 240
dock     -> door    : 3.4 m
door     -> cabinet : 9.047 m
cabinet  -> corner  : 1.888 m
corner   -> dock    : 4.784 m


## 8. Lazy collision checking (Note, Section 8): the roadmap of Section 2, trip 2 (door -> cabinet)

In [16]:
wl = prm.World()
llog = []
PL, LL, checked, searches = prm.lazy_query(wl, V, prm.K, prm.DOOR, prm.CABINET, llog)
print('lazy: searches', searches, ' edges checked', checked, ' configurations checked', wl.checks, ' route', round(LL, 3))
for s_, path, key, ok in llog:
    print('  search', s_, 'edge', key, 'free' if ok else 'COLLIDES')
print('eager, same trip: phase 2', phase2, '+ query', trip_cost[1], '=', phase2 + trip_cost[1])
print('route on the eagerly checked roadmap', round(prm.query(prm.World(), V, E, prm.DOOR, prm.CABINET)[1], 3))

lazy: searches 5  edges checked 11  configurations checked 129  route 3.579
  search 1 edge (0, 100) free
  search 1 edge (0, 78) free
  search 1 edge (64, 78) free
  search 1 edge (64, np.int64(97)) COLLIDES
  search 2 edge (16, 100) COLLIDES
  search 3 edge (60, 100) COLLIDES
  search 4 edge (0, np.int64(60)) free
  search 4 edge (np.int64(60), np.int64(69)) free
  search 4 edge (np.int64(69), np.int64(97)) COLLIDES
  search 5 edge (np.int64(72), 78) free
  search 5 edge (np.int64(72), np.int64(101)) free
eager, same trip: phase 2 7745 + query 209 = 7954
route on the eagerly checked roadmap 3.587
